In [ ]:
# Problema: Mostrar cómo claves reales de telemetría pueden concentrar trabajo y cómo un combiner reduce pares antes del shuffle.

import csv
import gzip
import hashlib
from collections import defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data").is_dir() and (p / "submission").is_dir())
with gzip.open(ROOT / "data/truck_events.csv.gz", "rt", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
def partition(key): return int(hashlib.md5(str(key).encode()).hexdigest(), 16) % 4
balanced, skewed = [0] * 4, [0] * 4
for row in rows:
    balanced[partition(row["eventKey"])] += 1
    skewed[partition(row["eventType"])] += 1
partials = [defaultdict(int) for _ in range(4)]
for row in rows:
    partials[partition(row["eventType"])][row["eventType"]] += 1
combined_pairs = sum(len(item) for item in partials)
with (ROOT / "submission/partition_loads.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["partition", "event_key_load", "account_key_load"], lineterminator="\n")
    writer.writeheader(); writer.writerows({"partition": i, "event_key_load": balanced[i], "account_key_load": skewed[i]} for i in range(4))
with (ROOT / "submission/shuffle_comparison.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["raw_pairs", "pairs_after_local_combiner", "hot_key_load"], lineterminator="\n")
    writer.writeheader(); writer.writerow({"raw_pairs": len(rows), "pairs_after_local_combiner": combined_pairs, "hot_key_load": max(skewed)})